# Feature extraction — AutoEncoder

Theo Kabir, Chen & Ludwig, *A performance analysis of dimensionality reduction algorithms in machine learning models for cancer prediction*, Healthcare Analytics 3 (2023) 100125 (mục 3.2.1, Fig. 2):

- **Kiến trúc** (5 lớp ẩn, kể cả lớp latent): `Input(p) → 5000 → 550 → Latent → 550 → 5000 → Output(p)`; encoder = 3 lớp đầu, decoder đối xứng.
- **Hàm mất mát:** MSE tái tạo (nhãn của AE là chính đầu vào — học không giám sát, **không dùng label**). Bài báo đạt MSE tái tạo ≈ 0,5.
- **Sau khi train chỉ giữ encoder**; đầu ra lớp latent là đặc trưng mới. Bài báo đặt latent = 250 (bằng số PC của PCA); ở đây latent = `N_FEATURES` (mặc định 100) để so sánh cùng số chiều với các phương pháp khác.

Bài báo không nêu hàm kích hoạt, optimizer, số epoch — notebook chọn: ReLU ở lớp ẩn, latent và output tuyến tính, Adam (lr = 1e-3), batch 32, tối đa `AE_EPOCHS` epoch với **early stopping** theo MSE trên 10% tách từ **train** của fold (không dùng val/test), khôi phục trọng số tốt nhất. Mặc định `PREPROCESS = "none"` (giữ log2 gốc). **Tự dùng GPU nếu có** (Kaggle: Settings → Accelerator → GPU): với ~20k gene, lớp p × 5000 có ~100M tham số mỗi phía, CPU sẽ rất chậm.

- **Khác với feature selection:** đặc trưng mới là **tổ hợp của gene** (không còn là gene gốc). Mỗi fold: **fit chỉ trên `train`**, sau đó dùng mô hình đã fit để biến đổi `val` và `test` (val/test không tham gia fit). Dữ liệu đầu vào giữ nguyên log2 expression, không chuẩn hoá (trừ khi ghi rõ bên dưới).
- **Đầu vào:** `DATA_DIR/fold_{1..5}/{train,val,test}.h5`.
- **Đổi dataset:** sửa `DATA_DIR` ở cell cấu hình. Số chiều đầu ra: `N_FEATURES` (mặc định 100).
- **Đầu ra** (`/kaggle/working/feature_extraction/autoencoder/<dataset>/`):
  - `features/fold_k/{train,val,test}.h5` — dữ liệu sau trích xuất, **cùng format với dữ liệu gốc** (pandas, key=`"data"`, index = sample_id, cột = `AE1..AEk` + `label`). Notebook phân loại chỉ cần trỏ `DATA_DIR`/`DATA_ROOT` vào thư mục `features/`.
  - `components/fold_k.csv`, `components_all_folds.csv` — thông tin từng thành phần (rank, score: phương sai của từng chiều latent trên train (xếp giảm dần))
  - `metrics_per_fold.csv`, `metrics_summary.csv` — thời gian, CPU utilization, peak memory, energy, EDP, carbon footprint
- **Đo lường:** `ResourceTracker` chỉ bao quanh lời gọi `fit_extractor` (fit trên train + biến đổi train), không tính thời gian đọc file. Thời gian biến đổi val+test ghi riêng ở cột `transform_time_sec`. Energy/Carbon ước lượng bằng `codecarbon` (tracking_mode="process"); EDP = Energy(J) × Time(s).

In [ ]:
# tables: backend cho pd.read_hdf | codecarbon: do nang luong/CO2 | psutil: CPU & RAM
get_ipython().system("pip install -q codecarbon psutil tables")

In [ ]:
import os

# ====== CAU HINH (sua tay) ======
# Doi DATA_DIR sang bo khac (vd .../h5/tcga_luad) de chay dataset khac
DATA_DIR = "/kaggle/input/datasets/lfreedom2750/experiment-dataset/h5/gse68465"
N_FEATURES = 100          # so chieu dac trung moi sau trich xuat
N_FOLDS = 5
SEED = 42
METHOD = "autoencoder"
AE_HIDDEN = (5000, 550)       # encoder: p -> 5000 -> 550 -> latent (decoder doi xung), theo Fig. 2 bai bao
AE_ACTIVATION = "relu"        # bai bao khong neu; "relu" | "elu" | "tanh" | "sigmoid"
AE_LR = 1e-3
AE_BATCH = 32
AE_EPOCHS = 200               # so epoch toi da
AE_PATIENCE = 20              # early stopping theo MSE holdout
AE_HOLDOUT = 0.1              # ti le train tach ra lam holdout cho early stopping
AE_WEIGHT_DECAY = 0.0
PREPROCESS = "none"           # "none" (log2 goc) | "zscore" -- thong ke tinh tren train cua fold
METHOD_PARAMS = {"hidden": AE_HIDDEN, "activation": AE_ACTIVATION, "lr": AE_LR, "batch": AE_BATCH, "epochs": AE_EPOCHS, "patience": AE_PATIENCE, "holdout": AE_HOLDOUT, "weight_decay": AE_WEIGHT_DECAY, "loss": "mse", "optimizer": "adam", "preprocess": PREPROCESS, "reference": "Kabir et al., Healthcare Analytics 3 (2023) 100125"}
# None -> codecarbon tu dinh vi (can Internet); hoac dat ma ISO 3 ky tu, vd "USA", "VNM"
COUNTRY_ISO_CODE = None

DATASET_NAME = os.path.basename(os.path.normpath(DATA_DIR))
RESULT_DIR = os.path.join("/kaggle/working/feature_extraction", METHOD, DATASET_NAME)
os.makedirs(os.path.join(RESULT_DIR, "components"), exist_ok=True)

for _s in ["train", "val", "test"]:
    assert os.path.isfile(os.path.join(DATA_DIR, "fold_1", f"{_s}.h5")), \
        f"Khong thay {DATA_DIR}/fold_1/{_s}.h5 -- da Add Input dataset chua?"
print("Dataset :", DATASET_NAME, "->", DATA_DIR)
print("Method  :", METHOD, "| N_FEATURES =", N_FEATURES)
print("Ket qua :", RESULT_DIR)
print("So CPU  :", os.cpu_count())

## Đọc dữ liệu

In [ ]:
import json, time, threading, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

STAGE_NAMES = {0: "Normal", 1: "I", 2: "II", 3: "III", 4: "IV"}


def load_split(fold, split="train"):
    # File h5 (pandas, key="data"): index = sample_id, cot = ten gene + cot "label"
    df = pd.read_hdf(os.path.join(DATA_DIR, f"fold_{fold}", f"{split}.h5"), key="data")
    X = df.drop(columns="label")
    # Giu nguyen gia tri log2 expression, KHONG chuan hoa
    return X.values.astype(np.float64), df["label"].values.astype(int), X.columns.to_numpy(), df.index.to_numpy()


# Doc thu fold 1 de xem cau truc
X0, y0, genes0, ids0 = load_split(1)
print(f"fold_1/train: {X0.shape[0]} mau x {X0.shape[1]} gene | khoang gia tri [{X0.min():.2f}, {X0.max():.2f}]")
print("Vi du gene:", list(genes0[:5]), "| vi du sample:", list(ids0[:3]))
print("Phan bo lop:", pd.Series(y0).map(STAGE_NAMES).value_counts().to_dict())
del X0, y0, genes0, ids0

## Bộ đo tài nguyên

In [ ]:
import psutil

try:
    from codecarbon import EmissionsTracker, OfflineEmissionsTracker
    _CODECARBON = True
except ImportError:
    _CODECARBON = False


class ResourceTracker:
    """Do tren 1 khoi lenh: thoi gian (s), CPU utilization, peak memory (RSS, gom ca
    tien trinh con cua joblib), energy (kWh, codecarbon), carbon (kg CO2eq) va
    Energy-Delay Product EDP = Energy(J) * Time(s)."""

    def __init__(self, name, interval=0.05):
        self.name, self.interval = name, interval
        self.proc = psutil.Process()
        self._stop = threading.Event()

    def _procs(self):
        try:
            return [self.proc] + self.proc.children(recursive=True)
        except psutil.Error:
            return [self.proc]

    def _poll(self):
        # RSS tong va CPU time cua worker con (tru phan CPU worker da dung truoc khi bat dau do)
        rss = 0
        for p in self._procs():
            try:
                rss += p.memory_info().rss
                if p.pid != self.proc.pid:
                    t = p.cpu_times()
                    self._child_cpu[p.pid] = t.user + t.system - self._child_base.get(p.pid, 0.0)
            except psutil.Error:
                pass
        self._peak_rss = max(self._peak_rss, rss)
        return rss

    def _sampler(self):
        while not self._stop.is_set():
            self._poll()
            self._sys_cpu.append(psutil.cpu_percent(interval=None))
            self._stop.wait(self.interval)

    def __enter__(self):
        self._child_cpu, self._sys_cpu, self._peak_rss = {}, [], 0
        self._child_base = {}
        for p in self._procs()[1:]:
            try:
                t = p.cpu_times()
                self._child_base[p.pid] = t.user + t.system
            except psutil.Error:
                pass
        self.tracker = None
        if _CODECARBON:
            try:
                kw = dict(project_name=self.name, output_dir=RESULT_DIR, save_to_file=False,
                          log_level="error", measure_power_secs=1, tracking_mode="process")
                self.tracker = (OfflineEmissionsTracker(country_iso_code=COUNTRY_ISO_CODE, **kw)
                                if COUNTRY_ISO_CODE else EmissionsTracker(**kw))
                self.tracker.start()
            except Exception as e:
                print("[ResourceTracker] codecarbon loi:", e)
                self.tracker = None
        self.base_rss = self._poll()
        self._child_cpu = {}
        psutil.cpu_percent(interval=None)
        t = self.proc.cpu_times()
        self._cpu0 = t.user + t.system
        self._stop.clear()
        self._thread = threading.Thread(target=self._sampler, daemon=True)
        self._thread.start()
        self._t0 = time.perf_counter()
        return self

    def __exit__(self, *exc):
        self.time_sec = time.perf_counter() - self._t0
        t = self.proc.cpu_times()
        main_cpu = t.user + t.system - self._cpu0
        self._stop.set()
        self._thread.join(timeout=2)
        self._poll()
        cpu_sec = main_cpu + sum(self._child_cpu.values())

        self.energy_kwh, self.carbon_kg = float("nan"), float("nan")
        if self.tracker is not None:
            try:
                self.carbon_kg = float(self.tracker.stop() or 0.0)
                data = self.tracker.final_emissions_data
                self.energy_kwh = float(data.energy_consumed) if data is not None else float("nan")
            except Exception as e:
                print("[ResourceTracker] codecarbon stop loi:", e)

        n_cpu = os.cpu_count() or 1
        self.metrics = {
            "time_sec": self.time_sec,
            "cpu_time_sec": cpu_sec,
            # % tren tong nang luc may (100% = dung het n_cpu core)
            "cpu_util_percent": 100.0 * cpu_sec / (self.time_sec * n_cpu) if self.time_sec > 0 else float("nan"),
            "avg_cores_used": cpu_sec / self.time_sec if self.time_sec > 0 else float("nan"),
            "system_cpu_percent_mean": float(np.mean(self._sys_cpu)) if self._sys_cpu else float("nan"),
            "peak_memory_mb": self._peak_rss / 1024 ** 2,
            "peak_memory_increase_mb": (self._peak_rss - self.base_rss) / 1024 ** 2,
            "energy_kwh": self.energy_kwh,
            "energy_joule": self.energy_kwh * 3.6e6,
            "edp_joule_sec": self.energy_kwh * 3.6e6 * self.time_sec,
            "carbon_kg_co2eq": self.carbon_kg,
            "carbon_g_co2eq": self.carbon_kg * 1e3,
        }
        return False

## Thuật toán: AutoEncoder (Kabir et al., 2023)

In [ ]:
PREFIX = "AE"        # ten cot dac trung moi: AE1, AE2, ...
N_TOP_GENES = 20     # so gene top |loading| luu cho moi thanh phan
SELECT_INFO = {}   # thong tin phu cua lan fit gan nhat, duoc ghi vao metrics_per_fold.csv

def top_loading_genes(W, genes, comp_names, n_top):
    # W: (n_comp, n_gene) -> top n_top gene co |trong so| lon nhat cua moi thanh phan
    rows = []
    for c, w in zip(comp_names, np.asarray(W)):
        idx = np.argsort(-np.abs(w), kind="stable")[:n_top]
        rows += [{"component": c, "rank": r + 1, "gene": genes[i], "loading": float(w[i])}
                 for r, i in enumerate(idx)]
    return pd.DataFrame(rows)

import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

# Tu dung GPU neu co (Kaggle: Settings -> Accelerator -> GPU)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None
print("Thiet bi:", DEVICE, "|", GPU_NAME or f"{os.cpu_count()} CPU", "| torch", torch.__version__)
if DEVICE.type == "cuda":
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

_ACT = {"relu": nn.ReLU, "elu": nn.ELU, "tanh": nn.Tanh, "sigmoid": nn.Sigmoid}


class AutoEncoder(nn.Module):
    """Input(p) -> 5000 -> 550 -> latent -> 550 -> 5000 -> Output(p). Latent va output tuyen tinh."""

    def __init__(self, p, latent):
        super().__init__()
        enc, dec, d = [], [], p
        for h in AE_HIDDEN:
            enc += [nn.Linear(d, h), _ACT[AE_ACTIVATION]()]
            d = h
        enc.append(nn.Linear(d, latent))
        d = latent
        for h in reversed(AE_HIDDEN):
            dec += [nn.Linear(d, h), _ACT[AE_ACTIVATION]()]
            d = h
        dec.append(nn.Linear(d, p))
        self.encoder, self.decoder = nn.Sequential(*enc), nn.Sequential(*dec)

    def forward(self, x):
        return self.decoder(self.encoder(x))


class AEModel:
    """Encoder da train + thong ke tien xu ly (tinh tren train)."""

    def __init__(self, net, mu, sd):
        self.net, self.mu, self.sd = net, mu, sd

    def prep(self, X):
        return (X - self.mu) / self.sd

    @torch.no_grad()
    def encode(self, X, batch=256):
        self.net.eval()
        Xp = self.prep(X)
        out = [self.net.encoder(torch.tensor(Xp[s:s + batch], dtype=torch.float32, device=DEVICE)).cpu().numpy()
               for s in range(0, len(Xp), batch)]
        return np.vstack(out).astype(np.float64)

    @torch.no_grad()
    def recon_mse(self, X, batch=256):
        self.net.eval()
        Xp, se = self.prep(X), 0.0
        for s in range(0, len(Xp), batch):
            xb = torch.tensor(Xp[s:s + batch], dtype=torch.float32, device=DEVICE)
            se += F.mse_loss(self.net(xb), xb, reduction="sum").item()
        return se / Xp.size


def _gpu_begin():
    SELECT_INFO.clear()
    SELECT_INFO.update(device=DEVICE.type, gpu_name=GPU_NAME)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


def _gpu_end():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
        SELECT_INFO["gpu_peak_memory_mb"] = torch.cuda.max_memory_allocated() / 1024 ** 2


def resolve_k(X, y):
    return N_FEATURES


def fit_extractor(X, y, k):
    _gpu_begin()
    torch.manual_seed(SEED)
    if PREPROCESS == "zscore":
        mu, sd = X.mean(0), X.std(0)
        sd[sd == 0] = 1.0
    else:
        mu, sd = np.zeros(X.shape[1]), np.ones(X.shape[1])

    # Holdout tach tu TRAIN cua fold (khong dung val/test) cho early stopping; AE khong dung nhan
    fit_i, hold_i = train_test_split(np.arange(len(X)), test_size=AE_HOLDOUT, random_state=SEED)
    Xf_t = torch.tensor((X[fit_i] - mu) / sd, dtype=torch.float32, device=DEVICE)
    Xh_t = torch.tensor((X[hold_i] - mu) / sd, dtype=torch.float32, device=DEVICE)

    net = AutoEncoder(X.shape[1], k).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=AE_LR, weight_decay=AE_WEIGHT_DECAY)
    best, best_state, bad, ep = np.inf, None, 0, 0
    for ep in range(1, AE_EPOCHS + 1):
        net.train()
        perm = torch.randperm(len(Xf_t), device=DEVICE)
        for s in range(0, len(perm), AE_BATCH):
            xb = Xf_t[perm[s:s + AE_BATCH]]
            opt.zero_grad(set_to_none=True)
            F.mse_loss(net(xb), xb).backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            h = F.mse_loss(net(Xh_t), Xh_t).item()
        if h < best - 1e-6:
            best, best_state, bad = h, copy.deepcopy(net.state_dict()), 0
        else:
            bad += 1
        if ep % 20 == 0:
            print(f"    epoch {ep:4d} | holdout MSE {h:.4f} | best {best:.4f}")
        if bad >= AE_PATIENCE:
            break
    net.load_state_dict(best_state)
    del Xf_t, Xh_t, opt

    model = AEModel(net, mu, sd)
    Z = model.encode(X)
    _gpu_end()
    var = Z.var(0)
    order = np.argsort(-var, kind="stable")   # sap xep chieu latent theo phuong sai giam dan
    model.order = order
    SELECT_INFO.update(epochs_run=ep, best_holdout_mse=float(best), recon_mse_train=model.recon_mse(X),
                       n_params=sum(p.numel() for p in net.parameters()))
    return model, Z[:, order], var[order]


def transform(model, X):
    return model.encode(X)[:, model.order]


def loadings(model):
    return None

## Chạy trên 5 fold

In [ ]:
all_rows, metric_rows, loading_rows = [], [], []
VIS = {}   # luu fold 1 de ve hinh

for fold in range(1, N_FOLDS + 1):
    X, y, genes, ids = load_split(fold, "train")   # CHI dung tap train de fit
    Xv, yv, genes_v, ids_v = load_split(fold, "val")
    Xt, yt, genes_t, ids_t = load_split(fold, "test")
    assert (genes_v == genes).all() and (genes_t == genes).all(), "Thu tu gene train/val/test khac nhau"
    k = resolve_k(X, y)

    with ResourceTracker(f"{METHOD}_{DATASET_NAME}_fold{fold}") as rt:
        model, Z, scores = fit_extractor(X, y, k)

    t0 = time.perf_counter()
    Zv, Zt = transform(model, Xv), transform(model, Xt)
    transform_time = time.perf_counter() - t0

    names = [f"{PREFIX}{i}" for i in range(1, Z.shape[1] + 1)]
    out_dir = os.path.join(RESULT_DIR, "features", f"fold_{fold}")
    os.makedirs(out_dir, exist_ok=True)
    for split, Zs, ys, idx_s in [("train", Z, y, ids), ("val", Zv, yv, ids_v), ("test", Zt, yt, ids_t)]:
        df = pd.DataFrame(np.asarray(Zs, dtype=np.float64), index=idx_s, columns=names)
        df["label"] = ys
        df.to_hdf(os.path.join(out_dir, f"{split}.h5"), key="data", mode="w")

    comp = pd.DataFrame({"fold": fold, "rank": np.arange(1, len(names) + 1), "component": names, "score": scores})
    comp.to_csv(os.path.join(RESULT_DIR, "components", f"fold_{fold}.csv"), index=False)
    all_rows.append(comp)
    W = loadings(model)
    if W is not None:
        loading_rows.append(top_loading_genes(W, genes, names, N_TOP_GENES).assign(fold=fold))

    m = {"dataset": DATASET_NAME, "method": METHOD, "fold": fold,
         "n_train": X.shape[0], "n_val": Xv.shape[0], "n_test": Xt.shape[0],
         "n_genes_total": X.shape[1], "n_components": Z.shape[1],
         "transform_time_sec": transform_time, **SELECT_INFO, **rt.metrics}
    metric_rows.append(m)
    print(f"fold {fold}: {X.shape} -> {Z.shape[1]} chieu | {m['time_sec']:.2f}s | CPU {m['cpu_util_percent']:.1f}% "
          f"({m['avg_cores_used']:.2f} core) | peak {m['peak_memory_mb']:.0f} MB | "
          f"{m['energy_kwh']:.3e} kWh | {m['carbon_g_co2eq']:.4f} gCO2eq | transform {transform_time:.2f}s")
    if fold == 1:
        VIS = {"train": (np.asarray(Z), y), "test": (np.asarray(Zt), yt), "names": names}
    del X, Xv, Xt, model

components = pd.concat(all_rows, ignore_index=True)
components.to_csv(os.path.join(RESULT_DIR, "components_all_folds.csv"), index=False)
if loading_rows:
    pd.concat(loading_rows, ignore_index=True).to_csv(os.path.join(RESULT_DIR, "loadings_top_genes.csv"), index=False)

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(os.path.join(RESULT_DIR, "metrics_per_fold.csv"), index=False)
metrics

## Tổng hợp độ đo

In [ ]:
cols = ["time_sec", "cpu_time_sec", "cpu_util_percent", "avg_cores_used", "system_cpu_percent_mean",
        "peak_memory_mb", "peak_memory_increase_mb", "energy_kwh", "energy_joule",
        "edp_joule_sec", "carbon_kg_co2eq", "carbon_g_co2eq", "transform_time_sec"]
summary = metrics[cols].agg(["mean", "std", "min", "max"]).T
summary.loc["time_sec", "total"] = metrics["time_sec"].sum()
summary.loc["energy_kwh", "total"] = metrics["energy_kwh"].sum()
summary.loc["carbon_kg_co2eq", "total"] = metrics["carbon_kg_co2eq"].sum()
summary.to_csv(os.path.join(RESULT_DIR, "metrics_summary.csv"))

with open(os.path.join(RESULT_DIR, "config.json"), "w") as f:
    json.dump({"dataset": DATASET_NAME, "data_dir": DATA_DIR, "method": METHOD, "n_features": N_FEATURES,
               "n_folds": N_FOLDS, "seed": SEED, "params": METHOD_PARAMS, "cpu_count": os.cpu_count(),
               "country_iso_code": COUNTRY_ISO_CODE}, f, indent=1, default=str)
summary

## Trực quan hoá không gian mới (tham khảo)

In [ ]:
# Tham khao nhanh: 2 chieu dau tien cua fold 1 (train: cham tron, test: dau x), to mau theo stage
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 5))
names = VIS["names"]
if len(names) >= 2:
    for split, marker, alpha in [("train", "o", 0.45), ("test", "x", 0.9)]:
        Zs, ys = VIS[split]
        for c in np.unique(ys):
            mk = ys == c
            ax.scatter(Zs[mk, 0], Zs[mk, 1], marker=marker, s=18, alpha=alpha,
                       color=plt.cm.tab10(int(c)), label=f"{STAGE_NAMES.get(int(c), c)} ({split})")
    ax.set_xlabel(names[0]); ax.set_ylabel(names[1])
else:   # chi co 1 chieu (vd LDA 2 lop) -> ve phan bo 1 chieu
    for split, marker in [("train", "o"), ("test", "x")]:
        Zs, ys = VIS[split]
        ax.scatter(Zs[:, 0], ys + (0.15 if split == "test" else 0), marker=marker, c=ys, cmap="tab10", s=18)
    ax.set_xlabel(names[0]); ax.set_ylabel("label")
ax.set_title(f"{METHOD} | {DATASET_NAME} | fold 1")
ax.legend(fontsize=7, ncol=2, loc="best")
fig.tight_layout()
fig.savefig(os.path.join(RESULT_DIR, "scatter_fold1.png"), dpi=150)
plt.show()

In [ ]:
import shutil
zip_path = shutil.make_archive(f"/kaggle/working/fe_{METHOD}_{DATASET_NAME}", "zip", RESULT_DIR)
print("Da nen:", zip_path)
for root, _, files in os.walk(RESULT_DIR):
    for fn in sorted(files):
        print(" ", os.path.relpath(os.path.join(root, fn), RESULT_DIR))